# This notebook trains a CNN for each cross-user and cross-environment study.


# Cross user

In [1]:
import sys
sys.path.append('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/src')

In [2]:
# Load datasets using ThermalDataset (not the slow Aggregator)
from dataset.dataset import ThermalDataset
from posture_detection_module.utils import filter_dataset, train_model, ThermalNormalize
from posture_detection_module.CNN_model import SimpleIRA_CNN
from torch.utils.data import Subset, ConcatDataset
from sklearn.model_selection import train_test_split
import torch

In [3]:
# load the yaml file containing experiment setup
# path: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/config/exp_setup.yaml
def get_exp_training_testing_setup(env_name):
    ret = {
        'train': [],
        'test': []
    }
    import yaml
    with open('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/config/exp_setup.yaml', 'r') as f:
        exp_setup = yaml.safe_load(f)
    
    envs = [f"env{i}" for i in range(0, 6)]
    for env in envs:
        if env != env_name:
            ret['train'].extend(exp_setup[env])
        else:
            ret['test'].extend(exp_setup[env])
    
    return ret

print(list(filename.split("/")[-1] for filename in get_exp_training_testing_setup("env5")['train']))
print(list(filename.split("/")[-1] for filename in get_exp_training_testing_setup("env5")['test']))


['hall5', 'home01', 'home2', 'home31', 'office0_4', 'office0_3', 'office0_2', 'office0_0']
['office1_0']


In [4]:

from pathlib import Path
import sys
sys.path.insert(0, "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/src")

from dataset.dataset import ThermalDatasetAggregator
from posture_detection_module.CNN_model import SimpleIRA_CNN

In [5]:
# env 6 cross-env model training
model_path = "posture_cnn_cross_env_env6_0417.pth" # mistake: wrote to 0
# load the datasets for training and testing
train_test = get_exp_training_testing_setup("env6")
train_path_lst = train_test['train']
test_path_lst = train_test['test']

kept_labels = [2, 3, 4, 5, 6]
label_to_index = {label: idx for idx, label in enumerate(kept_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

In [6]:
# Build list of filtered datasets from train paths
train_datasets = []
all_entries = []
all_labels = []

for path in train_path_lst:
    ds = ThermalDataset(path)
    ds_filtered = filter_dataset(ds, label_to_index)
    train_datasets.append(ds_filtered)
    print(f"Loaded {path.split('/')[-1]}: {len(ds_filtered)} samples")
    for i in range(len(ds_filtered)):
        all_entries.append((ds_filtered, i))
        all_labels.append(ds_filtered.dataset.annotations_expanded[ds_filtered.indices[i]])

print(f"\nTotal training samples: {len(all_entries)}")

test_datasets = []
for path in test_path_lst:
    ds = ThermalDataset(path)
    ds_filtered = filter_dataset(ds, label_to_index)
    test_datasets.append(ds_filtered)
    
print(f"Loaded {path.split('/')[-1]}: {len(ds_filtered)} samples")



Loaded hall5: 1769 samples
Loaded home01: 3901 samples
Loaded home2: 1858 samples
Loaded home31: 1694 samples
Loaded office0_4: 3907 samples
Loaded office0_3: 6012 samples
Loaded office0_2: 11480 samples
Loaded office0_0: 1837 samples
Loaded office1_0: 3815 samples

Total training samples: 36273
Loaded office1_0: 3815 samples


In [8]:
# Stratified train/val split
train_entries, val_entries = train_test_split(
    all_entries, test_size=0.2, stratify=all_labels, random_state=42
)

# Create dataloaders
def make_loader(entries, batch_size=32, augment=False):
    indices_by_ds = {}
    for ds, i in entries:
        if ds not in indices_by_ds:
            indices_by_ds[ds] = []
        indices_by_ds[ds].append(i)
    subsets = [Subset(ds, indices) for ds, indices in indices_by_ds.items()]
    concat = ConcatDataset(subsets) if len(subsets) > 1 else subsets[0]
    return torch.utils.data.DataLoader(
        ThermalNormalize(concat, augment=augment), batch_size=batch_size, shuffle=True
    )

train_loader = make_loader(train_entries, augment=True)
val_loader = make_loader(val_entries, augment=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Create and train model
model = SimpleIRA_CNN(num_classes=len(kept_labels))
train_model(model, train_loader, val_loader, label_to_index, num_epochs=15, learning_rate=1e-3, save_path=model_path)

# Save model
# save_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_env_env5_0417.pth'
# torch.save(model.state_dict(), save_path)
# print(f"Model saved to {save_path}")

Train batches: 907, Val batches: 227


Epoch 1/15: 100%|██████████| 907/907 [06:13<00:00,  2.43it/s]


Epoch 1/15, Loss: 0.1262
Validation Accuracy: 0.9928
Model saved to posture_cnn_cross_env_env6_0417.pth with loss 0.1262


Epoch 2/15: 100%|██████████| 907/907 [07:11<00:00,  2.10it/s]


Epoch 2/15, Loss: 0.0330
Validation Accuracy: 0.9968
Model saved to posture_cnn_cross_env_env6_0417.pth with loss 0.0330


...

Epoch 13/15, Loss: 0.0048
Validation Accuracy: 0.9999
Model saved to posture_cnn_cross_env_env6_0417.pth with loss 0.0048


Epoch 14/15: 100%|██████████| 907/907 [03:19<00:00,  4.56it/s]


Epoch 14/15, Loss: 0.0080
Validation Accuracy: 0.9999


Epoch 15/15: 100%|██████████| 907/907 [03:08<00:00,  4.80it/s]


Epoch 15/15, Loss: 0.0060
Validation Accuracy: 1.0000


In [ ]:
# Test model on test dataset

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/src")

# Load datasets using ThermalDataset (not the slow Aggregator)
from dataset.dataset import ThermalDataset
from posture_detection_module.utils import filter_dataset, train_model, ThermalNormalize
from posture_detection_module.CNN_model import SimpleIRA_CNN
from torch.utils.data import Subset, ConcatDataset
from sklearn.model_selection import train_test_split
import torch

# load the yaml file containing experiment setup
# path: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/config/exp_setup.yaml
def get_exp_training_testing_setup(env_name):
    ret = {
        'train': [],
        'test': []
    }
    import yaml
    with open('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/config/exp_setup.yaml', 'r') as f:
        exp_setup = yaml.safe_load(f)
    
    envs = [f"env{i}" for i in range(0, 6)]
    for env in envs:
        if env != env_name:
            ret['train'].extend(exp_setup[env])
        else:
            ret['test'].extend(exp_setup[env])
    
    return ret

env_name = "env5"

# env 5 cross-env model training
model_path = f"posture_cnn_cross_env_{env_name}_0417.pth"
# load the datasets for training and testing
train_test = get_exp_training_testing_setup(env_name)
train_path_lst = train_test['train']
test_path_lst = train_test['test']

kept_labels = [2, 3, 4, 5, 6]
label_to_index = {label: idx for idx, label in enumerate(kept_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

# ==================================================
# Build list of filtered datasets from train paths
train_datasets = []
all_entries = []
all_labels = []

for path in train_path_lst:
    ds = ThermalDataset(path)
    ds_filtered = filter_dataset(ds, label_to_index)
    train_datasets.append(ds_filtered)
    print(f"Loaded {path.split('/')[-1]}: {len(ds_filtered)} samples")
    for i in range(len(ds_filtered)):
        all_entries.append((ds_filtered, i))
        all_labels.append(ds_filtered.dataset.annotations_expanded[ds_filtered.indices[i]])

print(f"\nTotal training samples: {len(all_entries)}")

test_datasets = []
for path in test_path_lst:
    ds = ThermalDataset(path)
    ds_filtered = filter_dataset(ds, label_to_index)
    test_datasets.append(ds_filtered)
    
print(f"Loaded {path.split('/')[-1]}: {len(ds_filtered)} samples")

# ==================================================

# Stratified train/val split
train_entries, val_entries = train_test_split(
    all_entries, test_size=0.2, stratify=all_labels, random_state=42
)

# Create dataloaders
def make_loader(entries, batch_size=32, augment=False):
    indices_by_ds = {}
    for ds, i in entries:
        if ds not in indices_by_ds:
            indices_by_ds[ds] = []
        indices_by_ds[ds].append(i)
    subsets = [Subset(ds, indices) for ds, indices in indices_by_ds.items()]
    concat = ConcatDataset(subsets) if len(subsets) > 1 else subsets[0]
    return torch.utils.data.DataLoader(
        ThermalNormalize(concat, augment=augment), batch_size=batch_size, shuffle=True
    )

train_loader = make_loader(train_entries, augment=True)
val_loader = make_loader(val_entries, augment=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Create and train model
model = SimpleIRA_CNN(num_classes=len(kept_labels))
train_model(model, train_loader, val_loader, label_to_index, num_epochs=15, learning_rate=1e-3, save_path=model_path)

# Save model
# save_path = f'/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_env_{env_name}_0417.pth'
# torch.save(model.state_dict(), save_path)
# print(f"Model saved to {save_path}")

Loaded hall5: 1769 samples
Loaded home01: 3901 samples
Loaded home2: 1858 samples
Loaded home31: 1694 samples
Loaded office0_4: 3907 samples
Loaded office0_3: 6012 samples
Loaded office0_2: 11480 samples
Loaded office0_0: 1837 samples

Total training samples: 32458
Loaded office1_0: 3815 samples
Train batches: 812, Val batches: 203


Epoch 1/15:   6%|▌         | 48/812 [00:11<03:05,  4.12it/s]


KeyboardInterrupt: 

In [ ]:
# use the data from: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/case_study_hall_exit_ref
# to fine-tune the model
# 1. load state dict from posture_cnn_cross_env_env5_0417.pth

# 2. load dataset, filter it, and create dataloader

# 3. train model for 1 epoch on the new dataset

# 4. save it as posture_cnn_cross_casestudy_0417.pth

from dataset.dataset import ThermalDataset
from posture_detection_module.utils import filter_dataset, train_model, ThermalNormalize
from posture_detection_module.CNN_model import SimpleIRA_CNN
from torch.utils.data import Subset, ConcatDataset
from sklearn.model_selection import train_test_split
import torch
import numpy as np

# ==== 1. Load pretrained weights ====
pretrained_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/weights_final/posture_cnn_cross_env_env5_0417.pth'
model = SimpleIRA_CNN(num_classes=5)
model.load_state_dict(torch.load(pretrained_path))
print(f"Loaded pretrained weights from {pretrained_path}")

# ==== 2. Load case study dataset ====
case_study_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/case_study_hall_exit_ref'
ds = ThermalDataset(case_study_path, noCam=True)

kept_labels = [2, 3, 4, 5, 6]
label_to_index = {label: idx for idx, label in enumerate(kept_labels)}
ds_filtered = filter_dataset(ds, label_to_index)
print(f"Loaded case study dataset: {len(ds_filtered)} samples")

# Filter to only samples with valid (non-None) ira_highres data
valid_indices = []
for i in range(len(ds_filtered)):
    ira = ds_filtered.dataset.get_ira_highres(ds_filtered.indices[i])
    if ira is not None and isinstance(ira, np.ndarray):
        valid_indices.append(i)

ds_filtered = Subset(ds_filtered, valid_indices)
print(f"After filtering None values: {len(ds_filtered)} samples")

# Get the root dataset (ThermalDataset, not Subset)
root_ds = ds_filtered.dataset
while isinstance(root_ds, Subset):
    root_ds = root_ds.dataset

# Create entries list for DataLoader
all_entries = [(ds_filtered, i) for i in range(len(ds_filtered))]
all_labels = [root_ds.annotations_expanded[ds_filtered.indices[i]] for i in range(len(ds_filtered))]

# Split into train/val
train_entries, val_entries = train_test_split(
    all_entries, test_size=0.2, stratify=all_labels, random_state=42
)

# Create dataloaders
def make_loader(entries, batch_size=32, augment=False):
    indices_by_ds = {}
    for ds, i in entries:
        if ds not in indices_by_ds:
            indices_by_ds[ds] = []
        indices_by_ds[ds].append(i)
    subsets = [Subset(ds, indices) for ds, indices in indices_by_ds.items()]
    concat = ConcatDataset(subsets) if len(subsets) > 1 else subsets[0]
    return torch.utils.data.DataLoader(
        ThermalNormalize(concat, augment=augment), batch_size=batch_size, shuffle=True
    )

train_loader = make_loader(train_entries, augment=True)
val_loader = make_loader(val_entries, augment=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# ==== 3. Fine-tune for 1 epoch ====
save_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0417.pth'
train_model(model, train_loader, val_loader, label_to_index, num_epochs=1, learning_rate=1e-4, save_path=save_path)

# ==== 4. Save fine-tuned model ====
torch.save(model.state_dict(), save_path)
print(f"Fine-tuned model saved to {save_path}")

Loaded pretrained weights from /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/weights_final/posture_cnn_cross_env_env5_0417.pth
Loaded case study dataset: 898 samples
After filtering None values: 898 samples
Train batches: 23, Val batches: 6


Epoch 1/1: 100%|██████████| 23/23 [00:04<00:00,  5.69it/s]


Epoch 1/1, Loss: 0.1027
Validation Accuracy: 1.0000
Model saved to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0417.pth with loss 0.1027
Fine-tuned model saved to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0417.pth


In [ ]:
# use the data from: /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/sleep_casestudy1
# to fine-tune the model
# 1. load state dict from posture_cnn_cross_env_env5_0417.pth

# 2. load dataset, filter it, and create dataloader

# 3. train model for 1 epoch on the new dataset

# 4. save it as posture_cnn_cross_casestudy_0417.pth

from dataset.dataset import ThermalDataset
from posture_detection_module.utils import filter_dataset, train_model, ThermalNormalize
from posture_detection_module.CNN_model import SimpleIRA_CNN
from torch.utils.data import Subset, ConcatDataset
from sklearn.model_selection import train_test_split
import torch
import numpy as np

# ==== 1. Load pretrained weights ====
pretrained_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0417.pth'
model = SimpleIRA_CNN(num_classes=5)
model.load_state_dict(torch.load(pretrained_path))
print(f"Loaded pretrained weights from {pretrained_path}")

# ==== 2. Load case study dataset ====
case_study_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/sleep_casestudy1'
ds = ThermalDataset(case_study_path, noCam=True)

kept_labels = [2, 3, 4, 5, 6]
label_to_index = {label: idx for idx, label in enumerate(kept_labels)}
ds_filtered = filter_dataset(ds, label_to_index)
print(f"Loaded case study dataset: {len(ds_filtered)} samples")

# Filter to only samples with valid (non-None) ira_highres data
valid_indices = []
for i in range(len(ds_filtered)):
    ira = ds_filtered.dataset.get_ira_highres(ds_filtered.indices[i])
    if ira is not None and isinstance(ira, np.ndarray):
        valid_indices.append(i)

ds_filtered = Subset(ds_filtered, valid_indices)
print(f"After filtering None values: {len(ds_filtered)} samples")

# Get the root dataset (ThermalDataset, not Subset)
root_ds = ds_filtered.dataset
while isinstance(root_ds, Subset):
    root_ds = root_ds.dataset

# Create entries list for DataLoader
all_entries = [(ds_filtered, i) for i in range(len(ds_filtered))]
all_labels = [root_ds.annotations_expanded[ds_filtered.indices[i]] for i in range(len(ds_filtered))]

# Split into train/val
train_entries, val_entries = train_test_split(
    all_entries, test_size=0.2, stratify=all_labels, random_state=42
)

# Create dataloaders
def make_loader(entries, batch_size=32, augment=False):
    indices_by_ds = {}
    for ds, i in entries:
        if ds not in indices_by_ds:
            indices_by_ds[ds] = []
        indices_by_ds[ds].append(i)
    subsets = [Subset(ds, indices) for ds, indices in indices_by_ds.items()]
    concat = ConcatDataset(subsets) if len(subsets) > 1 else subsets[0]
    return torch.utils.data.DataLoader(
        ThermalNormalize(concat, augment=augment), batch_size=batch_size, shuffle=True
    )

train_loader = make_loader(train_entries, augment=True)
val_loader = make_loader(val_entries, augment=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# ==== 3. Fine-tune for 1 epoch ====
save_path = '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0419.pth'
train_model(model, train_loader, val_loader, label_to_index, num_epochs=2, learning_rate=5e-4, save_path=save_path)

# ==== 4. Save fine-tuned model ====
torch.save(model.state_dict(), save_path)
print(f"Fine-tuned model saved to {save_path}")

Loaded pretrained weights from /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0417.pth
Loaded case study dataset: 2698 samples
After filtering None values: 2698 samples
Train batches: 68, Val batches: 17


Epoch 1/2: 100%|██████████| 68/68 [00:12<00:00,  5.49it/s]


Epoch 1/2, Loss: 0.7255
Validation Accuracy: 0.9500
Model saved to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0419.pth with loss 0.7255


Epoch 2/2: 100%|██████████| 68/68 [00:10<00:00,  6.30it/s]


Epoch 2/2, Loss: 0.1449
Validation Accuracy: 0.9796
Model saved to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0419.pth with loss 0.1449
Fine-tuned model saved to /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn_cross_casestudy_0419.pth
